# Huấn luyện và Fine-tune PhoBERT cho phân tích cảm xúc bình luận YouTube
**Dự án:** YTB_sentiment_analysis  
**Thành viên phụ trách:** Bạn Hùng (Deep Learning & PhoBERT)  
**Mục tiêu:** Cấu hình PhoBERT Tokenizer, Fine-tune PhoBERT trên GPU Google Colab, đánh giá chỉ số ($Accuracy, F1$) so sánh với SVM, xuất file trọng số lưu lên Google Drive chung.

## 1. Kiểm tra cấu hình GPU và Cài đặt môi trường

In [ ]:
!nvidia-smi
!pip install -q transformers torch accelerate scikit-learn pandas numpy matplotlib seaborn underthesea pyvi tqdm

## 2. Clone Repository Git của nhóm & Tự động nạp dữ liệu sạch chuẩn UTF-8
Toàn bộ code và tập dữ liệu `train.csv`, `test.csv` sẽ được tự động tải và trích xuất chuẩn UTF-8 ngay lập tức (không cần upload thủ công).

In [ ]:
import os
import pandas as pd

# 1. Clone repository nếu chưa clone
if not os.path.exists('/content/YTB_sentiment_analysis'):
    !git clone https://github.com/dung06coder/YTB_sentiment_analysis.git

%cd /content/YTB_sentiment_analysis
!git checkout feature/hung-phobert-dl
!git pull origin feature/hung-phobert-dl

# 2. Tự động khôi phục dữ liệu gốc UTF-8 chuẩn từ git history
!git checkout 815cecc -- machine_learning/data
!git reset HEAD machine_learning/data

# 3. Kiểm tra đọc dữ liệu
df_train = pd.read_csv('machine_learning/data/train.csv', encoding='utf-8')
df_test = pd.read_csv('machine_learning/data/test.csv', encoding='utf-8')
print("\n✅ Dữ liệu đã sẵn sàng 100%:")
print(f"- Số lượng mẫu tập Train: {len(df_train):,}")
print(f"- Số lượng mẫu tập Test : {len(df_test):,}")
display(df_train.head(3))

## 3. Tiến hành Huấn luyện (Fine-tune) PhoBERT trên GPU (CUDA + FP16)
Chạy script `train_phobert.py` đã được cấu hình tối ưu với:
- PhoBERT Base (`vinai/phobert-base`)
- Tokenizer hỗ trợ từ tiếng Việt có phân đoạn từ (`segment=True`)
- Batch size 32, Learning Rate 2e-5, AdamW + Linear Warmup
- Tự động chọn checkpoint tốt nhất theo Macro F1

In [ ]:
!python machine_learning/train_phobert.py \
    --train machine_learning/data/train.csv \
    --test machine_learning/data/test.csv \
    --epochs 4 \
    --batch-size 32 \
    --lr 2e-5 \
    --max-length 128 \
    --fp16

## 4. Đánh giá và Vẽ biểu đồ so sánh hiệu năng với SVM Baseline
Chạy `evaluate_and_compare.py` để xuất:
1. Biểu đồ so sánh Accuracy & F1-score (`comparison_metrics_barchart.png`)
2. Biểu đồ F1 từng lớp (`f1_per_class_comparison.png`)
3. Ma trận nhầm lẫn PhoBERT (`phobert_confusion_matrix.png`)
4. Bảng mã nguồn LaTeX chép vào báo cáo đồ án (`latex_comparison_table.tex`)
5. Báo cáo phân tích chuyên sâu (`comparison_report.md`)

In [ ]:
!python machine_learning/evaluate_and_compare.py

# Hiển thị biểu đồ trực quan ngay trên Colab
from IPython.display import Image, display
display(Image('machine_learning/reports/comparison_metrics_barchart.png'))
display(Image('machine_learning/reports/f1_per_class_comparison.png'))
display(Image('machine_learning/reports/phobert_confusion_matrix.png'))

## 5. Nén trọng số mô hình và tải về / lưu vào Google Drive chung
> **LƯU Ý NGHIÊM NGẶT:** File trọng số PhoBERT (`.safetensors` dung lượng ~540MB) **TUYỆT ĐỐI CẤM** đẩy lên GitHub. Hãy nén và đẩy vào Google Drive chung của nhóm để bạn làm Backend tải về tích hợp.

In [ ]:
from google.colab import drive

# Nén thư mục model đã fine-tune
!zip -r /content/phobert_finetuned.zip machine_learning/models/phobert_finetuned

# Mount Drive và sao chép vào thư mục chung của nhóm
try:
    drive.mount('/content/drive')
    DRIVE_MODELS_DIR = '/content/drive/MyDrive/YTB_sentiment_analysis/models'
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    !cp /content/phobert_finetuned.zip {DRIVE_MODELS_DIR}/
    print(f"\n✅ Đã lưu thành công mô hình lên Google Drive: {DRIVE_MODELS_DIR}/phobert_finetuned.zip")
except Exception as e:
    print(f"Không thể kết nối Drive tự động: {e}")
    print("Bạn có thể bấm vào tab Files bên trái Colab và tải file '/content/phobert_finetuned.zip' về máy!")

## 6. Kiểm thử nhanh hàm suy luận (Inference Demo)

In [ ]:
from machine_learning.predict_phobert import PhoBertSentimentPredictor

predictor = PhoBertSentimentPredictor('machine_learning/models/phobert_finetuned')

sample_comments = [
    "Video phân tích quá chi tiết và hữu ích, cảm ơn anh nhiều! <3",
    "Nội dung nhảm nhí, giật tít câu view phí thời gian 😡",
    "Bài hát ở phút 3:45 tên là gì vậy mọi người?",
    "Lúc đầu xem tưởng dở ai dè hay vãi chưởng kkkk"
]

for cmt in sample_comments:
    res = predictor.predict_comment(cmt)
    print(f"Bình luận: {res['text']}")
    print(f"--> Cảm xúc: {res['label'].upper()} (Độ tin cậy: {res['confidence']:.2%})")
    print(f"    Xác suất: {res['probabilities']}\n")